In [1]:
#! pip install tqdm multiprocess

In [41]:
import numpy as np
import pandas as pd
from multiprocess import Pool
import time 

from functions import evaluate_ar, evaluate_multiplier_iid, evaluate_ma
from bstrapping.synthetic_time_series.moving_average import MovingAverage
from bstrapping.bootstrap_procedures.weighted_bootstrap import WeightedBootstrap
from bstrapping.weights.auto_regressive_weights import AutoRegressiveWeights
from bstrapping.weights.moving_average import MovingAverageWeights
from bstrapping.weights.gaussian_weights import GaussianWeights

In [42]:
mean = 4  # mean of the time series
alpha = 0.1

# Names of the stochastic processes

# Dependence coefficients of the stochastic processes, i.e. of the moving average processes
dependence_coefficient = np.array([0.5 ** i for i in range(1, 3)])

list_name_weights = ['AR',
                     'Multiplier',
                     'MA',
                     ]

In [43]:
runs = 50

## Benchmark bootstraps

In [44]:
sample_sizes = [1000,2000,5000,10000]

In [45]:
%%time
%%capture

p = Pool()
durations_ar_mean = []
durations_ar_std = []
for sample_size in sample_sizes:
    duration_ar = []
    sample = MovingAverage(mean=mean, 
                           parameters=dependence_coefficient).generate_samples(sample_size)
    for _ in range(runs):
        start_time = time.perf_counter()
        WeightedBootstrap(samples=sample,
                          weights=AutoRegressiveWeights(samples=sample),
                          number_bootstrap_samples=20)
        end_time = time.perf_counter()
        duration_ar.append(end_time - start_time)
    
    durations_ar_mean.append(np.mean(duration_ar))
    durations_ar_std.append(np.std(duration_ar))


CPU times: user 1min 3s, sys: 88.4 ms, total: 1min 3s
Wall time: 1min 46s


In [46]:
%%time
%%capture

p = Pool()
durations_iid_mean = []
durations_iid_std = []
for sample_size in sample_sizes:
    duration_iid = []
    sample = MovingAverage(mean=mean, 
                           parameters=dependence_coefficient).generate_samples(sample_size)
    for _ in range(runs):
        start_time = time.perf_counter()
        WeightedBootstrap(samples=sample,
                          weights=GaussianWeights(samples=sample),
                          number_bootstrap_samples=20)
        end_time = time.perf_counter()
        duration_iid.append(end_time - start_time)
    
    durations_iid_mean.append(np.mean(duration_iid))
    durations_iid_std.append(np.std(duration_iid))

CPU times: user 752 ms, sys: 109 ms, total: 861 ms
Wall time: 2.46 s


In [47]:
%%time
%%capture

p = Pool()
durations_ma_mean = []
durations_ma_std = []
for sample_size in sample_sizes:
    duration_ma = []
    sample = MovingAverage(mean=mean, 
                           parameters=dependence_coefficient).generate_samples(sample_size)
    for _ in range(runs):
        start_time = time.perf_counter()
        WeightedBootstrap(samples=sample,
                          weights=MovingAverageWeights(samples=sample),
                          number_bootstrap_samples=20)
        end_time = time.perf_counter()
        duration_ma.append(end_time - start_time)
    
    durations_ma_mean.append(np.mean(duration_ma))
    durations_ma_std.append(np.std(duration_ma))

CPU times: user 18min 47s, sys: 269 ms, total: 18min 47s
Wall time: 26min


## Concatination result

In [48]:
benchmark = pd.DataFrame(
    [durations_ar_mean,durations_iid_mean,durations_ma_mean,
    durations_ar_std,durations_iid_std,durations_ma_std],
    columns=sample_sizes,
    index= pd.MultiIndex.from_tuples([("mean","AR"),("mean","Multiplier"),("mean","MA"),
                                     ("std","AR"),("std","Multiplier"),("std","MA")])
).T

In [49]:
benchmark

mean                             std                     
             AR Multiplier         MA        AR Multiplier        MA
1000   0.080856   0.002845   1.676740  0.033698   0.006072  0.442920
2000   0.199021   0.005019   3.629806  0.067649   0.008375  0.682730
5000   0.725943   0.010197   6.911740  0.299773   0.011207  2.398458
10000  1.121932   0.014549  18.980911  0.363703   0.010392  6.687736

In [ ]:
benchmark.to_csv(f"./data/benchmark_times.csv")
benchmark.to_pickle(f"./data/benchmark_times.pkl")

In [ ]:
benchmark